# 🚦 Notebook 3: Parking Lot — Real-world extensions

Interviewers often follow up with *"ok, now what if…?"* questions. This notebook
shows three common follow-ups, each building on the "best" version from notebook 2:

1. **Concurrency** — two cars arrive at different entrances at the exact same moment.
2. **Reserved spots** — EV-charging, handicapped, or compact-only spots.
3. **Payment processing** — swap cash, credit-card, or mobile-app backends.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/parking-lot
uv sync
```

Select the `.venv` kernel. If not visible, reload: `Cmd+Shift+P` → **Reload Window**.

We'll redefine the classes here so this notebook is self-contained.

In [ ]:
from __future__ import annotations
from enum import Enum
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import itertools, threading, time

class Size(Enum):
    MOTORCYCLE = 1
    CAR        = 2
    TRUCK      = 3

@dataclass
class Vehicle:
    plate: str
    size:  Size
    is_ev: bool = False               # new flag for the EV extension

class Spot:
    _ids = itertools.count(1)
    def __init__(self, size: Size, *, ev_charger: bool = False, handicapped: bool = False):
        self.id          = next(Spot._ids)
        self.size        = size
        self.ev_charger  = ev_charger
        self.handicapped = handicapped
        self.vehicle     = None

    def can_fit(self, v: Vehicle) -> bool:
        if self.vehicle is not None: return False
        if v.size.value > self.size.value: return False
        return True

    def park(self, v: Vehicle):  self.vehicle = v
    def leave(self):             self.vehicle = None

    def __repr__(self):
        tags = []
        if self.ev_charger:  tags.append('EV')
        if self.handicapped: tags.append('HC')
        who = 'free' if self.vehicle is None else self.vehicle.plate
        tag_str = ('[' + ','.join(tags) + ']') if tags else ''
        return f'Spot#{self.id}({self.size.name}{tag_str},{who})'

@dataclass
class Ticket:
    vehicle:    Vehicle
    spot:       Spot
    entry_time: float = field(default_factory=time.time)


## 1. Concurrency — two cars, one spot

Imagine two entrances. Both threads check "is spot #7 free?" at the same time,
both see `True`, and both try to park. Without locking, **two vehicles appear to
occupy the same spot** — one gets silently overwritten.

Let's reproduce the race, then fix it with a single lock on the lot.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

class UnsafeLot:
    """No locks. Expect occasional double-booking under contention."""
    def __init__(self, spots): self.spots = spots
    def park(self, v: Vehicle):
        for s in self.spots:
            if s.can_fit(v):
                # Artificial scheduler pause between check and assignment
                time.sleep(0.0005)
                s.park(v)
                return Ticket(v, s)
        raise RuntimeError('full')

def _try_park(lot, i):
    try:    lot.park(Vehicle(f'P-{i}', Size.CAR))
    except RuntimeError: pass

def race(lot_factory, trials=50, workers=8):
    """Run many parks in parallel; count trials where some workers got overwritten."""
    overwrites = 0
    for _ in range(trials):
        lot = lot_factory()
        with ThreadPoolExecutor(max_workers=workers) as ex:
            list(ex.map(lambda i: _try_park(lot, i), range(workers)))
        parked_plates = [s.vehicle.plate for s in lot.spots if s.vehicle]
        # With >= workers free spots, every worker should get its own spot.
        # If fewer distinct plates than workers ended up parked, someone was overwritten.
        if len(parked_plates) < workers:
            overwrites += 1
    return overwrites

unsafe_factory = lambda: UnsafeLot([Spot(Size.CAR) for _ in range(8)])
print('unsafe races with overwrites (out of 50 trials):', race(unsafe_factory))


In [ ]:
class SafeLot:
    """Guarded by a single lock — no double-booking."""
    def __init__(self, spots):
        self.spots  = spots
        self._lock  = threading.Lock()
    def park(self, v: Vehicle):
        with self._lock:                           # only one thread at a time
            for s in self.spots:
                if s.can_fit(v):
                    time.sleep(0.0005)
                    s.park(v)
                    return Ticket(v, s)
            raise RuntimeError('full')

safe_factory = lambda: SafeLot([Spot(Size.CAR) for _ in range(8)])
print('safe   races with overwrites (out of 50 trials):', race(safe_factory))


> 🧠 The unsafe lot occasionally has fewer parked vehicles than attempts, because
> two threads raced and overwrote each other. The safe lot never does.
>
> In production you'd prefer **per-spot locks** (or a lock-free compare-and-set on
> each spot) to avoid a single-lock bottleneck — but for an interview, showing that
> you **recognize the race** and **name the fix** is often enough.


## 2. Reserved spots (EV-charging, handicapped)

Adding flags to `Spot` was the easy part. The real design question is
*"how do we route the right vehicle to the right spot?"* without copy-pasting logic.

We'll introduce a small **Rule** abstraction so admins can combine constraints.

In [ ]:
class SpotRule(ABC):
    @abstractmethod
    def allows(self, spot: Spot, v: Vehicle) -> bool: ...

class SizeRule(SpotRule):
    def allows(self, spot, v): return v.size.value <= spot.size.value

class EvOnlyRule(SpotRule):
    """EV-charger spots are reserved for EV vehicles."""
    def allows(self, spot, v):
        return (not spot.ev_charger) or v.is_ev

class AllRules(SpotRule):
    """Composite: every inner rule must allow the vehicle."""
    def __init__(self, *rules): self.rules = rules
    def allows(self, spot, v):  return all(r.allows(spot, v) for r in self.rules)

class SmartLot:
    def __init__(self, spots, rule: SpotRule):
        self.spots, self.rule = spots, rule
        self._lock = threading.Lock()

    def park(self, v: Vehicle) -> Ticket:
        with self._lock:
            for s in self.spots:
                if s.vehicle is None and self.rule.allows(s, v):
                    s.park(v)
                    return Ticket(v, s)
            raise RuntimeError('no eligible spot')

# Three spots: a normal car spot, an EV-charging car spot, a handicapped car spot
spots = [
    Spot(Size.CAR),
    Spot(Size.CAR, ev_charger=True),
    Spot(Size.CAR, handicapped=True),
]
lot = SmartLot(spots, rule=AllRules(SizeRule(), EvOnlyRule()))

gasoline = Vehicle('GAS-1', Size.CAR, is_ev=False)
electric = Vehicle('TES-1', Size.CAR, is_ev=True)

t1 = lot.park(gasoline)
t2 = lot.park(electric)
print('gas parked at:', t1.spot)
print('EV  parked at:', t2.spot)


Notice the gasoline car was **blocked** from the EV spot by the rule, so the
EV car could claim it. Adding a "compact-only" rule is now a **new class**, not an
edit to `SmartLot` — that's Open/Closed in action.

## 3. Pluggable payment

Just like pricing was a strategy, **collecting** the money is its own strategy.
That way unit tests use a fake, production uses Stripe, the kiosk uses cash, etc.

In [ ]:
class PaymentProcessor(ABC):
    @abstractmethod
    def charge(self, amount: float, reference: str) -> bool: ...

class CashRegister(PaymentProcessor):
    def __init__(self): self.total = 0.0
    def charge(self, amount, reference):
        self.total += amount
        print(f'cash ${amount:.2f} collected for {reference}')
        return True

class CreditCardGateway(PaymentProcessor):
    def __init__(self, fail_for=None):
        self.fail_for = fail_for or set()
    def charge(self, amount, reference):
        if reference in self.fail_for:
            print(f'card declined for {reference}')
            return False
        print(f'card ${amount:.2f} charged for {reference}')
        return True

class MockPayment(PaymentProcessor):
    """Useful in tests — records calls, never fails."""
    def __init__(self): self.calls = []
    def charge(self, amount, reference):
        self.calls.append((amount, reference))
        return True

class HourlyPricing:
    def __init__(self, rates): self.rates = rates
    def fee(self, ticket, now):
        hours = max(1, (now - ticket.entry_time) / 3600)
        return round(self.rates[ticket.vehicle.size] * hours, 2)

class FullLot(SmartLot):
    def __init__(self, spots, rule, pricing, payment: PaymentProcessor):
        super().__init__(spots, rule)
        self.pricing, self.payment = pricing, payment

    def leave(self, t: Ticket, now=None) -> float:
        amount = self.pricing.fee(t, now or time.time())
        ok = self.payment.charge(amount, reference=t.vehicle.plate)
        if not ok:
            # keep the spot occupied until payment succeeds
            raise RuntimeError(f'payment declined for {t.vehicle.plate}')
        t.spot.leave()
        return amount


In [ ]:
pricing = HourlyPricing({Size.MOTORCYCLE:1, Size.CAR:2, Size.TRUCK:4})
cash    = CashRegister()
card    = CreditCardGateway(fail_for={'BAD-1'})

lot_cash = FullLot(
    spots   = [Spot(Size.CAR), Spot(Size.CAR)],
    rule    = SizeRule(),
    pricing = pricing,
    payment = cash,
)

lot_card = FullLot(
    spots   = [Spot(Size.CAR), Spot(Size.CAR)],
    rule    = SizeRule(),
    pricing = pricing,
    payment = card,
)

# Happy-path cash
t = lot_cash.park(Vehicle('C-1', Size.CAR))
lot_cash.leave(t, now=t.entry_time + 3600)
print('register total: $', cash.total)

# Card success
t = lot_card.park(Vehicle('C-2', Size.CAR))
lot_card.leave(t, now=t.entry_time + 3600)

# Card decline -> the lot refuses to let them leave (spot stays occupied)
t = lot_card.park(Vehicle('BAD-1', Size.CAR))
try:
    lot_card.leave(t, now=t.entry_time + 3600)
except RuntimeError as e:
    print('expected:', e, '-- spot still occupied?', t.spot.vehicle is not None)


## 🎯 Recap — what a great interview answer looks like

1. **Clarify** scope and assumptions.
2. **Name your entities** and their relationships (notebook 1's diagram).
3. **Start simple**, then **refactor** to pluggable strategies (notebook 2's bad→best arc).
4. When asked "what about X?", reach for **composition**:
   - concurrency → `threading.Lock`
   - special spots → a `SpotRule` hierarchy
   - payments → a `PaymentProcessor` strategy
5. Call out which **SOLID** principle each change respects.

### Ideas to try on your own

- **Closest-fit** spot search (minimize walking) — change `Level.find_spot`.
- **Per-spot locks** instead of one lot-wide lock — reduce contention.
- **Subscriptions / monthly passes** — implement the `PricingStrategy` from notebook 2.
- **Display board** (`available()`) that pushes updates via an observer pattern.
- **Persistence** — save tickets to SQLite so the lot survives a restart.


## 🧪 Verify the design

Every claim above should be checkable. These assertions fail loudly if a
future edit breaks one of the three extensions.

In [ ]:
def must_raise(exc, fn, *a, **kw):
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__}, nothing was raised')

# --- 1. Concurrency: the lock actually removes the race -------------------
assert race(lambda: SafeLot([Spot(Size.CAR) for _ in range(8)]), trials=10) == 0, \
    'the locked lot must never double-book'

# --- 2. Rules: composite rule = AND of its parts -------------------------
ev_spot   = Spot(Size.CAR, ev_charger=True)
plain     = Spot(Size.CAR)
gas       = Vehicle('G', Size.CAR, is_ev=False)
tesla     = Vehicle('E', Size.CAR, is_ev=True)
rule      = AllRules(SizeRule(), EvOnlyRule())

assert not rule.allows(ev_spot, gas),   'EV-only spot must reject a gasoline car'
assert     rule.allows(ev_spot, tesla), 'EV spot must accept an EV'
assert     rule.allows(plain,   gas),   'a plain car spot accepts any car'
assert not rule.allows(plain, Vehicle('T', Size.TRUCK)), 'size rule still applies'

# Composition, not editing: a brand-new rule needs zero changes to SmartLot.
class CompactOnlyRule(SpotRule):
    def allows(self, spot, v): return v.size is not Size.TRUCK
strict = SmartLot([Spot(Size.TRUCK)], rule=AllRules(SizeRule(), CompactOnlyRule()))
must_raise(RuntimeError, strict.park, Vehicle('BIG', Size.TRUCK))

# --- 3. Payments: a declined charge must NOT release the spot -------------
declining = FullLot([Spot(Size.CAR)], SizeRule(), pricing,
                    CreditCardGateway(fail_for={'NOPE'}))
tk = declining.park(Vehicle('NOPE', Size.CAR))
must_raise(RuntimeError, declining.leave, tk, now=tk.entry_time + 3600)
assert tk.spot.vehicle is not None, 'declined payment must leave the spot occupied'

# ...and a successful charge must release it and record the money exactly once.
reg = CashRegister()
ok_lot = FullLot([Spot(Size.CAR)], SizeRule(), pricing, reg)
tk2 = ok_lot.park(Vehicle('OK-1', Size.CAR))
paid = ok_lot.leave(tk2, now=tk2.entry_time + 7200)   # 2h car @ $2/h
assert paid == 4.0 and reg.total == 4.0, (paid, reg.total)
assert tk2.spot.vehicle is None, 'paid vehicle must free its spot'

print('all extension invariants hold ✅')